# RobustSec quick start

This notebook trains an MLP detector on NSL-KDD and evaluates it under an unconstrained and a feasibility-aware threat model. It runs on the free Colab CPU runtime in a few minutes.

In [ ]:
!pip -q install git+https://github.com/aliqah0a/robustsec.git

In [ ]:
import robustsec as rs

ds = rs.load_nsl_kdd(download=True)
enc = rs.TabularEncoder(ds.schema).fit(ds.X_train)
X_tr, X_te = enc.transform(ds.X_train), enc.transform(ds.X_test)
clf = rs.fit_classifier(X_tr, ds.y_train, epochs=10, seed=0)
rs.clean_report(clf, X_te, ds.y_test)

## Threat models

The feasible threat model lets the attacker increase only `duration`, `src_bytes`, `dst_bytes`, `count` and `srv_count`. Every adversarial input is repaired to a valid NSL-KDD record.

In [ ]:
tms = {
    'unconstrained': rs.ThreatModel.unconstrained(enc.n_features_),
    'feasible': rs.ThreatModel.from_encoder(enc, rs.nsl_kdd_feasible_schema()),
}
rows = []
for tname, tm in tms.items():
    for eps in [0.02, 0.05, 0.1, 0.2]:
        X_adv = rs.PGD(eps, steps=30, threat_model=tm).generate(clf, X_te, ds.y_test)
        r = rs.robustness_report(clf, X_te, X_adv, ds.y_test)
        rows.append(dict(threat_model=tname, eps=eps, robust_acc=r['robust_accuracy'],
                         asr=r['asr'], evasion=r['evasion_rate'], l0=r['l0']))
import pandas as pd
pd.DataFrame(rows).round(3)

## Inspect adversarial records

`to_raw` returns records in the original feature space.

In [ ]:
X_adv = rs.PGD(0.1, steps=30, threat_model=tms['feasible']).generate(clf, X_te, ds.y_test)
changed = (X_adv != X_te).any(axis=1)
before, after = enc.to_raw(X_te[changed][:5]), enc.to_raw(X_adv[changed][:5])
cols = rs.data.NSL_KDD_FEASIBLE_MUTABLE
pd.concat({'before': before[cols], 'after': after[cols]}, axis=1)